# Full fine-tuning Pythia-410M step5000 on Alpaca

This notebook creates a controlled **before/after** experiment. It loads the early `EleutherAI/pythia-410m` checkpoint at revision `step5000`, evaluates and samples it unchanged, fully fine-tunes all parameters on exactly 10,000 cleaned Alpaca examples for one epoch, and repeats the same evaluation.

> `step5000` is only partially pretrained. This is an educational instruction-tuning experiment, not a recipe for a production chatbot. No Hugging Face token, adapter library, quantization, or external experiment tracker is used.

## 1. Install the pinned package stack

Run this first in a fresh Kaggle session. `%pip` installs into the active notebook environment.

In [ ]:
%pip install -q -U \
    "transformers==4.57.1" \
    "datasets==4.4.1" \
    "accelerate==1.11.0" \
    "trl==0.26.2"

## 2. Imports, reproducibility, and GPU precision

A CUDA GPU is required. BF16 is selected only when **every visible GPU** has compute capability 8.0 or newer; otherwise the notebook uses FP16. TF32 is enabled on Ampere-or-newer GPUs.

In [ ]:
import math
import re
import shutil
from pathlib import Path

import accelerate
import datasets
import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers
import trl
from datasets import load_dataset
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed
from trl import SFTConfig, SFTTrainer

SEED = 42
MODEL_ID = "EleutherAI/pythia-410m"
MODEL_REVISION = "step5000"
DATASET_ID = "yahma/alpaca-cleaned"
OUTPUT_DIR = "/kaggle/working/pythia-410m-step5000-alpaca-runs"
FINAL_DIR = "/kaggle/working/pythia-410m-step5000-alpaca-final"
MAX_LENGTH = 384
TRAIN_SIZE = 10_000
VALIDATION_SIZE = 256
DATASET_NUM_PROC = 2

set_seed(SEED)
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")
print(f"TRL: {trl.__version__}")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not found. In Kaggle, open Settings -> Accelerator and select a GPU "
        "(T4 or P100), then restart and run the notebook from the top."
    )

GPU_COUNT = torch.cuda.device_count()
capabilities = [torch.cuda.get_device_capability(i) for i in range(GPU_COUNT)]
AMPERE_OR_NEWER = all(major >= 8 for major, _ in capabilities)
USE_BF16 = AMPERE_OR_NEWER
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = AMPERE_OR_NEWER
torch.backends.cudnn.allow_tf32 = AMPERE_OR_NEWER

print(f"CUDA version: {torch.version.cuda}")
print(f"Visible GPU count: {GPU_COUNT}")
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    allocated = torch.cuda.memory_allocated(i) / 1024**2
    reserved = torch.cuda.memory_reserved(i) / 1024**2
    print(
        f"GPU {i}: {props.name} | compute capability {capabilities[i][0]}.{capabilities[i][1]} | "
        f"initial allocated {allocated:.1f} MiB | initial reserved {reserved:.1f} MiB"
    )
print(f"Selected training dtype: {MODEL_DTYPE}")
print(f"TF32 enabled: {AMPERE_OR_NEWER}")

## 3. Load the tokenizer and early model checkpoint

The tokenizer comes from the main model repository. The weights explicitly use `revision="step5000"`, so this is not the final Pythia base checkpoint. The repository is public and ungated.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
).to("cuda")

for config in (model.config, model.generation_config):
    config.pad_token_id = tokenizer.pad_token_id
    config.bos_token_id = tokenizer.bos_token_id
    config.eos_token_id = tokenizer.eos_token_id

print(f"Loaded: {MODEL_ID} @ {MODEL_REVISION}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model parameter dtype: {next(model.parameters()).dtype}")
print(
    "Special token IDs | "
    f"PAD={model.config.pad_token_id}, BOS={model.config.bos_token_id}, "
    f"EOS={model.config.eos_token_id}"
)

## 4. Prepare fixed train and validation splits

Empty outputs are removed before shuffling. With seed 42, the first 256 usable shuffled rows form a fixed held-out validation set; the next 10,000 form the training set. Each row becomes a standard Alpaca `prompt` plus a separate `completion`, and EOS is appended to every completion.

In [ ]:
raw_dataset = load_dataset(DATASET_ID, split="train")
usable_dataset = raw_dataset.filter(
    lambda row: bool((row.get("output") or "").strip()),
    num_proc=DATASET_NUM_PROC,
)
shuffled_dataset = usable_dataset.shuffle(seed=SEED)
required_rows = VALIDATION_SIZE + TRAIN_SIZE
if len(shuffled_dataset) < required_rows:
    raise ValueError(
        f"Need at least {required_rows:,} non-empty examples, found {len(shuffled_dataset):,}."
    )

validation_raw = shuffled_dataset.select(range(VALIDATION_SIZE))
train_raw = shuffled_dataset.select(range(VALIDATION_SIZE, required_rows))
EOS_TOKEN = tokenizer.eos_token

def to_prompt_completion(row):
    instruction = (row.get("instruction") or "").strip()
    input_text = (row.get("input") or "").strip()
    if input_text:
        prompt = (
            "Below is an instruction that describes a task, paired with an input that provides "
            "further context. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )
    else:
        prompt = (
            "Below is an instruction that describes a task. Write a response that appropriately "
            "completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            "### Response:\n"
        )
    completion = (row.get("output") or "").strip() + EOS_TOKEN
    return {"prompt": prompt, "completion": completion}

train_dataset = train_raw.map(
    to_prompt_completion,
    remove_columns=train_raw.column_names,
    num_proc=DATASET_NUM_PROC,
)
validation_dataset = validation_raw.map(
    to_prompt_completion,
    remove_columns=validation_raw.column_names,
    num_proc=DATASET_NUM_PROC,
)

assert len(train_dataset) == TRAIN_SIZE
assert len(validation_dataset) == VALIDATION_SIZE
assert all(text.endswith(EOS_TOKEN) for text in train_dataset["completion"])
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(validation_dataset):,}")
print(f"Remaining columns: {train_dataset.column_names}")

In [ ]:
example = train_dataset[0]
print("FULLY FORMATTED EXAMPLE")
print("-" * 80)
print(example["prompt"] + example["completion"])
print("-" * 80)
print(f"Completion ends with EOS: {example['completion'].endswith(EOS_TOKEN)}")

## 5. Configure full-parameter SFT

This is **completion-only loss**. Prompt tokens remain visible as context, but their labels are masked, so cross-entropy is calculated only on response tokens and the appended EOS. TRL supports this when the dataset keeps separate `prompt` and `completion` fields; `completion_only_loss=True` makes the choice explicit.

No `peft_config` is passed. All model parameters remain trainable. Gradient checkpointing reduces activation memory, so the model cache is disabled during training.

In [ ]:
test_questions = [
    {"question": "Explain photosynthesis in exactly two sentences.", "input": ""},
    {"question": 'Translate “The weather is beautiful today” into French.', "input": ""},
    {"question": "Calculate the average speed for 180 km in 3 hours.", "input": ""},
    {"question": "Write a Python function that checks whether an integer is prime.", "input": ""},
    {"question": "Give exactly three concise job interview tips.", "input": ""},
    {"question": "Explain why leaves appear green in simple language.", "input": ""},
]

def format_test_prompt(question, input_text=""):
    row = {"instruction": question, "input": input_text, "output": "placeholder"}
    return to_prompt_completion(row)["prompt"]

In [ ]:
class LossPrinterCallback(TrainerCallback):
    """Print compact training metrics every 10 optimizer steps."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if state.global_step > 0 and state.global_step % 10 == 0 and "loss" in logs:
            grad_norm = logs.get("grad_norm")
            grad_text = "n/a" if grad_norm is None else f"{float(grad_norm):.4f}"
            print(
                f"step {state.global_step:5d} | loss: {float(logs['loss']):.4f} | "
                f"lr: {float(logs.get('learning_rate', 0.0)):.6f} | grad_norm: {grad_text}"
            )

model.config.use_cache = False
sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_torch",
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=not USE_BF16,
    bf16=USE_BF16,
    tf32=AMPERE_OR_NEWER,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    dataset_num_proc=DATASET_NUM_PROC,
    seed=SEED,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[LossPrinterCallback()],
)

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
trainable_percentage = 100 * trainable_parameters / total_parameters
effective_batch_size = (
    sft_args.per_device_train_batch_size
    * sft_args.gradient_accumulation_steps
    * GPU_COUNT
)
estimated_optimizer_steps = math.ceil(len(train_dataset) / effective_batch_size)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,} ({trainable_percentage:.2f}%)")
if trainable_percentage < 99.0:
    raise RuntimeError("Fewer than 99% of parameters are trainable; this is not full fine-tuning.")
print("Confirmed: nearly all parameters are enabled for full fine-tuning.")
print(
    "Effective batch size: "
    f"{sft_args.per_device_train_batch_size} per device × "
    f"{sft_args.gradient_accumulation_steps} accumulation × {GPU_COUNT} GPU(s) "
    f"= {effective_batch_size}"
)
print(f"Estimated optimizer steps for one epoch: {estimated_optimizer_steps:,}")

## 6. Baseline evaluation and generation before training

The trainer already exists, but `trainer.train()` has not run. First we evaluate the untouched step5000 weights on the fixed validation set, then generate deterministic answers. Only newly generated tokens are decoded, so the stored answers do not repeat the prompt.

In [ ]:
baseline_metrics = trainer.evaluate()
baseline_eval_loss = float(baseline_metrics["eval_loss"])
print(f"Baseline validation loss: {baseline_eval_loss:.6f}")

In [ ]:
def generate_answers(current_model, questions):
    current_model.eval()
    device = next(current_model.parameters()).device
    answers = []
    for item in questions:
        prompt = format_test_prompt(item["question"], item.get("input", ""))
        encoded = tokenizer(prompt, return_tensors="pt")
        encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
        prompt_length = encoded["input_ids"].shape[1]
        with torch.inference_mode():
            generated = current_model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=128,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = generated[0, prompt_length:]
        answers.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return answers

baseline_answers = generate_answers(model, test_questions)
for item, answer in zip(test_questions, baseline_answers):
    print(f"QUESTION: {item['question']}")
    print(f"BEFORE: {answer}\n")

## 7. Train for one epoch

Training loss is measured on changing mini-batches, so it can move up and down and does **not** need to decrease monotonically. The fixed validation loss, evaluated every 100 optimizer steps, is the correct comparison for evidence of improvement. We will not claim success until final validation loss is compared with the saved baseline.

The checkpoint was loaded and baseline-tested in the selected half dtype. Immediately before optimization, its trainable parameters are promoted to FP32 master weights; the requested `fp16`/`bf16` flag still makes forward and backward computation use mixed precision. This standard AMP pattern prevents FP16 gradient-unscaling errors on P100/T4-class hardware while keeping every parameter trainable.

In [ ]:
# AMP safely updates FP32 master parameters while computing in the selected lower precision.
trainer.model.float()
torch.cuda.empty_cache()
print(
    f"Trainable master-weight dtype: {next(trainer.model.parameters()).dtype}; "
    f"AMP compute dtype: {MODEL_DTYPE}"
)
train_output = trainer.train()
print(train_output)
training_log_history = list(trainer.state.log_history)

## 8. Inspect loss and gradient logs

The two panels use separate scales for readability. The baseline validation loss is a dashed reference line; intermediate validation points come from the same fixed 256 examples.

In [ ]:
log_df = pd.DataFrame(training_log_history)
requested_columns = [
    "step",
    "loss",
    "eval_loss",
    "learning_rate",
    "grad_norm",
    "entropy",
    "mean_token_accuracy",
    "num_tokens",
]
available_columns = [column for column in requested_columns if column in log_df.columns]
display(log_df[available_columns].dropna(how="all").reset_index(drop=True))

train_rows = log_df.dropna(subset=["loss"]) if "loss" in log_df else pd.DataFrame()
eval_rows = log_df.dropna(subset=["eval_loss"]) if "eval_loss" in log_df else pd.DataFrame()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if not train_rows.empty:
    axes[0].plot(train_rows["step"], train_rows["loss"], linewidth=1.5)
else:
    axes[0].text(0.5, 0.5, "No training-loss rows found", ha="center")
axes[0].set(title="Training loss", xlabel="Optimizer step", ylabel="Loss")
axes[0].grid(alpha=0.25)

if not eval_rows.empty:
    axes[1].plot(
        eval_rows["step"], eval_rows["eval_loss"], marker="o", label="Validation loss"
    )
axes[1].axhline(
    baseline_eval_loss, color="tab:red", linestyle="--", label="Baseline validation loss"
)
axes[1].set(title="Fixed validation loss", xlabel="Optimizer step", ylabel="Loss")
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
logged_grad_norms = []
if "grad_norm" in log_df.columns:
    logged_grad_norms = pd.to_numeric(log_df["grad_norm"], errors="coerce").dropna().tolist()
nonzero_gradients_logged = any(value > 0 for value in logged_grad_norms)
print(f"Trainable parameter share before training: {trainable_percentage:.2f}%")
print(f"Nonzero gradient norm found in training logs: {nonzero_gradients_logged}")
if nonzero_gradients_logged:
    print("Confirmed: training logs contain nonzero gradients.")
else:
    print("No nonzero logged gradient norm was found; do not claim a verified update.")

## 9. Final evaluation on the same held-out set

Perplexity is `exp(loss)`. Very large or non-finite losses are mapped safely to infinity instead of overflowing. Positive absolute and percentage improvement mean that validation loss decreased.

In [ ]:
final_metrics = trainer.evaluate()
final_eval_loss = float(final_metrics["eval_loss"])
absolute_improvement = baseline_eval_loss - final_eval_loss
percentage_improvement = (
    100 * absolute_improvement / baseline_eval_loss
    if math.isfinite(baseline_eval_loss) and baseline_eval_loss != 0
    else float("nan")
)

def safe_perplexity(loss):
    if not math.isfinite(loss) or loss > 700:
        return float("inf")
    return math.exp(loss)

baseline_perplexity = safe_perplexity(baseline_eval_loss)
final_perplexity = safe_perplexity(final_eval_loss)
print(f"Baseline validation loss: {baseline_eval_loss:.6f}")
print(f"Final validation loss:    {final_eval_loss:.6f}")
print(f"Absolute improvement:     {absolute_improvement:.6f}")
print(f"Percentage improvement:   {percentage_improvement:.2f}%")
print(f"Baseline perplexity:      {baseline_perplexity:.4f}")
print(f"Final perplexity:         {final_perplexity:.4f}")

if final_eval_loss < baseline_eval_loss:
    print("Validation loss decreased; the model learned the held-out Alpaca distribution.")
else:
    print(
        "Validation loss did not decrease. Do not claim improvement; inspect the configuration "
        "and generated answers."
    )

## 10. Generate the same answers after training

Gradient checkpointing is disabled and the generation cache is restored. The function and decoding settings are exactly the same as before training.

In [ ]:
trainer.model.gradient_checkpointing_disable()
trainer.model.to(dtype=MODEL_DTYPE)
trainer.model.config.use_cache = True
trainer.model.generation_config.use_cache = True
trainer.model.eval()

finetuned_answers = generate_answers(trainer.model, test_questions)
comparison_df = pd.DataFrame(
    {
        "question": [item["question"] for item in test_questions],
        "optional_input": [item.get("input", "") for item in test_questions],
        "before_training": baseline_answers,
        "after_training": finetuned_answers,
    }
)
display(comparison_df)

## 11. Transparent answer checks and manual review

The checks below cover only four prompts with simple surface rules. They are intentionally transparent and limited: passing does not prove a response is correct, and failing can miss a valid phrasing. They are **not a substitute for human evaluation**.

In [ ]:
def speed_check(answer):
    return bool(re.search(r"\b60(?:\.0+)?\b", answer)) and "km" in answer.lower()

def french_check(answer):
    normalized = answer.lower().replace("’", "'")
    keywords = ["temps", "météo", "beau", "belle", "aujourd'hui"]
    return sum(keyword in normalized for keyword in keywords) >= 2

def prime_code_check(answer):
    normalized = answer.lower()
    has_function = bool(re.search(r"\bdef\s+\w+\s*\(", normalized))
    has_divisibility = "%" in answer or "mod" in normalized
    has_search_logic = any(token in normalized for token in ("range", "sqrt", "while"))
    return has_function and has_divisibility and has_search_logic

def three_item_check(answer):
    numbered = re.findall(r"(?m)^\s*[1-3][.)]\s+", answer)
    bullets = re.findall(r"(?m)^\s*[-*•]\s+", answer)
    return len(numbered) == 3 or len(bullets) == 3

rule_specs = [
    ("French translation", 1, french_check, "At least two plausible French keywords"),
    ("Average speed", 2, speed_check, 'Contains “60” and “km”'),
    ("Prime function", 3, prime_code_check, "Python function plus divisibility/search logic"),
    ("Interview tips", 4, three_item_check, "Exactly three numbered or bulleted items"),
]
rule_rows = []
for name, index, check, criterion in rule_specs:
    rule_rows.append(
        {
            "check": name,
            "criterion": criterion,
            "before_pass": bool(check(baseline_answers[index])),
            "after_pass": bool(check(finetuned_answers[index])),
        }
    )
rule_results_df = pd.DataFrame(rule_rows)
before_pass_count = int(rule_results_df["before_pass"].sum())
after_pass_count = int(rule_results_df["after_pass"].sum())
display(rule_results_df)
print(f"Before-training rule passes: {before_pass_count}/{len(rule_results_df)}")
print(f"After-training rule passes:  {after_pass_count}/{len(rule_results_df)}")
print("Reminder: these limited surface checks do not replace human evaluation.")

In [ ]:
manual_evaluation_df = pd.DataFrame(
    {
        "question": [item["question"] for item in test_questions],
        "instruction followed": [""] * len(test_questions),
        "factually correct": [""] * len(test_questions),
        "coherent": [""] * len(test_questions),
        "properly formatted": [""] * len(test_questions),
    }
)
print("Fill the blank fields after manually reviewing the after-training answers:")
display(manual_evaluation_df)

## 12. Save the complete fine-tuned model

This writes the full model weights, configuration, generation configuration, and tokenizer, then creates a ZIP archive. It is a full checkpoint—not a small LoRA adapter.

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

final_path = Path(FINAL_DIR)
archive_path = shutil.make_archive(
    str(final_path),
    "zip",
    root_dir=final_path.parent,
    base_dir=final_path.name,
)

print("Saved full-model files:")
for file_path in sorted(path for path in final_path.rglob("*") if path.is_file()):
    size_mib = file_path.stat().st_size / 1024**2
    print(f"  {file_path.relative_to(final_path)} — {size_mib:.2f} MiB")
archive_size_mib = Path(archive_path).stat().st_size / 1024**2
print(f"ZIP archive: {archive_path} — {archive_size_mib:.2f} MiB")
print("This archive contains a full fine-tuned model checkpoint, not a LoRA adapter.")

## 13. Memory fallback (only if CUDA runs out of memory)

Do not use the smaller settings preemptively. If the configured run raises a CUDA out-of-memory error, change only these values and rerun from the start:

```python
per_device_train_batch_size = 1
gradient_accumulation_steps = 16
max_length = 256
```

The effective single-GPU batch remains 16, while shorter sequences and a smaller micro-batch reduce peak VRAM.

## 14. Expected-result warning

`step5000` is only a partially pretrained checkpoint. Alpaca fine-tuning can teach recognizable instruction/response structure, but 10,000 supervised examples cannot replace the missing language pretraining. Even if held-out loss and the examples improve, expect weak knowledge, reasoning, factuality, safety behavior, and multi-turn conversation. Treat this as an educational before/after experiment—not as a production chatbot.